Membuat data target cabang dan data transaksi.

In [3]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota
data_target_cabang = {
    "kota": [
        "Magelang",
        "Yogyakarta",
        "Semarang",
        "Solo",
        "Purworejo"
    ],
    "target_bulanan": [
        45000000,
        60000000,
        55000000,
        40000000,
        30000000
    ],
    "pic_cabang": [
        "Rani",
        "Joko",
        "Sari",
        "Bayu",
        "Fitri"
    ],
}

# Tabel 2: Data transaksi
np.random.seed(55)

n = 500

kategori_list = [
    "Elektronik",
    "Fashion",
    "Makanan & Minuman",
    "Kesehatan & Kecantikan",
    "Rumah Tangga"
]

kota_list = [
    "Magelang",
    "Yogyakarta",
    "Semarang",
    "Solo",
    "Purworejo"
]

data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(
        kategori_list,
        size=n
    ),
    "kota": np.random.choice(
        kota_list,
        size=n
    ),
    "unit_terjual": np.random.randint(
        1,
        10,
        size=n
    ),
    "harga_satuan": np.random.choice(
        [
            25000,
            50000,
            75000,
            100000,
            150000
        ],
        size=n
    ),
}

pd.DataFrame(
    data_transaksi_t5
).to_csv(
    "transaksi_tugas5.csv",
    index=False
)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5

!hdfs dfs -put -f \
transaksi_tugas5.csv \
/user/mahasiswa/tugas5/

print(
    "Dataset siap. Tabel transaksi sudah diunggah ke HDFS:"
)

print(
    "/user/mahasiswa/tugas5/transaksi_tugas5.csv"
)

Dataset siap. Tabel transaksi sudah diunggah ke HDFS:
/user/mahasiswa/tugas5/transaksi_tugas5.csv


Buat SparkSession baru lalu baca file transaksi langsung dari HDFS.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    row_number,
    count
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas5") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv",
    header=True,
    inferSchema=True
)

df_transaksi = df_transaksi.withColumn(
    "pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df_target = spark.createDataFrame(
    pd.DataFrame(data_target_cabang)
)

print("Data transaksi:")
df_transaksi.show(5)

print("Data target cabang:")
df_target.show()

Data transaksi:
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

Data target cabang:
+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      550000

A. Join dan Perbandingan Target

In [5]:
ringkasan_kota = df_transaksi.groupBy(
    "kota"
).agg(
    spark_sum("pendapatan").alias(
        "total_pendapatan"
    )
)

hasil_A = ringkasan_kota.join(
    df_target,
    on="kota",
    how="inner"
)

hasil_A = hasil_A.withColumn(
    "pencapaian_persen",
    (
        col("total_pendapatan")
        / col("target_bulanan")
        * 100
    )
)

hasil_A.orderBy(
    col("pencapaian_persen").desc()
).show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



B. Kategori Terlaris perkota

In [6]:
pendapatan_kategori = df_transaksi.groupBy(
    "kota",
    "kategori"
).agg(
    spark_sum("pendapatan").alias(
        "total_pendapatan"
    )
)

window_kategori = Window \
    .partitionBy("kota") \
    .orderBy(
        col("total_pendapatan").desc()
    )

hasil_B = pendapatan_kategori.withColumn(
    "nomor_urut",
    row_number().over(window_kategori)
)

hasil_B.filter(
    col("nomor_urut") == 1
).orderBy("kota").show()

+----------+--------------------+----------------+----------+
|      kota|            kategori|total_pendapatan|nomor_urut|
+----------+--------------------+----------------+----------+
|  Magelang|Kesehatan & Kecan...|         7275000|         1|
| Purworejo|Kesehatan & Kecan...|        10075000|         1|
|  Semarang|        Rumah Tangga|        11125000|         1|
|      Solo|Kesehatan & Kecan...|         8425000|         1|
|Yogyakarta|             Fashion|        13325000|         1|
+----------+--------------------+----------------+----------+



C. SparkSQL

In [8]:
df_transaksi.createOrReplaceTempView(
    "transaksi"
)
df_target.createOrReplaceTempView(
    "target_cabang"
)

In [9]:
hasil_C = spark.sql('''
    SELECT
        t.kota,
        tc.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target_cabang tc
        ON t.kota = tc.kota
    GROUP BY
        t.kota,
        tc.pic_cabang
    ORDER BY
        jumlah_transaksi DESC
''')
hasil_C.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



D. Kesimpulan

Purworejo mempunyai pencapaian pendapatan sebesar 152,17% dari target bulanan. Total pendapatannya mencapai Rp45.650.000 dengan target sebesar Rp30.000.000. Dari hasil window function, kategori dengan pendapatan tertinggi di Purworejo adalah Kesehatan & Kecantikan dengan pendapatan Rp10.075.000. Sementara itu, Semarang mempunyai pencapaian sebesar 69,41% dengan total pendapatan Rp38.175.000 dari target Rp55.000.000. Kategori dengan pendapatan tertinggi di Semarang adalah Rumah Tangga sebesar Rp11.125.000. Dari hasil tersebut terlihat adanya perbedaan pencapaian antar cabang. Data ini bisa digunakan manajemen untuk melihat cabang yang sudah melewati target dan cabang yang masih mempunyai jarak cukup besar dari target.